# Notebook 05 — Genie Code and skills

**What you’ll do:** Use **Genie Code** with an optional **skill** so the assistant understands manufacturing terms (OEE, FPY, joins, table names).

**Two Genie spaces from 02:**
- **Main space** — instructions **plus** examples.
- **No-example space** — same instructions, **no** sample or curated SQL—useful for the comparison in **06**.

**Skill file (in this repo):** `skill/gm-genie-manufacturing-context/SKILL.md`.

**Before you start:** **01**–**04**. **Next:** **06** (compare spaces).


## Compute

Use **Serverless** (recommended).


## Open your Genie spaces

Use the links below for the **main** space and the **no-example** space (same link pattern as **02**).


In [ ]:
import html
import re

from databricks.sdk import WorkspaceClient

dbutils.widgets.text("catalog", "gm_ama_demos", "Catalog")
dbutils.widgets.text("schema", "genie_workshop_manufacturing", "Schema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
fqn = f"{CATALOG}.{SCHEMA}"


def genie_ui_room_url(host: str, space_id: str) -> str:
    h = (host or "").rstrip("/")
    sid = str(space_id or "").strip()
    m = re.search(r"adb-(\d+)\.", h)
    o = m.group(1) if m else ""
    q = f"?o={o}" if o else ""
    return f"{h}/genie/rooms/{sid}{q}"


w = WorkspaceClient()
_host = w.config.host.rstrip("/")

rows = spark.sql(
    f"""SELECT key, value, space_name FROM {fqn}.workshop_config
        WHERE key IN ('genie_space_id_configured_no_evals', 'genie_space_id')
        ORDER BY key"""
).collect()
for r in rows:
    _u = genie_ui_room_url(_host, r["value"])
    print(r["key"], "->", _u)
    displayHTML(
        "<p><b>"
        + html.escape(str(r["key"]))
        + "</b> &mdash; "
        + html.escape(str(r["space_name"]))
        + ' &mdash; <a href="'
        + html.escape(_u, quote=True)
        + '" target="_blank" rel="noopener">Open in Genie</a><br/><code style="font-size:11px">'
        + html.escape(_u)
        + "</code></p>"
    )


## Install skills (Genie Code)

Copy `skill/gm-genie-manufacturing-context/` into your user or repo **`.assistant/skills`** path (see current Genie Code docs). Skills are **not** stored inside the Genie SQL space JSON.


## Sample `SKILL.md` excerpt

Paste into the skill file or use the bundled copy in the repo.


In [ ]:
skill_md = """
---
name: gm-genie-manufacturing-context
description: Manufacturing quality metrics and joins for Genie Code
---

# GM manufacturing context

- OEE: `quality_metrics_daily.oee_score` is 0 to 1 (multiply by 100 for percent).
- FPY: `first_pass_yield` same scale.
- Defect rate: COUNT defect_detected / COUNT unit_produced on production_events.
- Join events to lines: production_events.production_line_id = production_lines.line_id
- Join lines to plants: production_lines.plant_id = plants.plant_id
"""
print(skill_md)
